# 玩家辨識 Baseline

這份 notebook 示範玩家辨識子題的完整流程：從官方提供的 CSV 讀進棋譜，抽出特徵、
訓練模型，最後產出可直接上傳的 `submission_player.csv`。

**這是參考實作，不是標準答案。** 模型架構、特徵設計、訓練方式都沒有限制，
歡迎自行修改。照本 notebook 跑完可以得到的分數列在最後一節，供你評估自己的改動
有沒有進步。

資料集與提交格式的正式定義以官方發佈為準，本 notebook 只補充「怎麼做」的部分。

## 任務

給定某位玩家的 5～20 局棋譜，從 400 位候選玩家中找出他是誰，交出最可能的 5 個人選。

## 環境需求

| 項目 | 需求 |
|---|---|
| GPU | 至少 4 GB VRAM（訓練約 2 GB，推論約 4 GB） |
| 磁碟 | 特徵約 15 GB，另需索引約 16 MB |
| 套件 | torch、sgfmill、pandas、numpy、matplotlib |

建議在具備 GPU 的 Linux 環境下執行。特徵抽取會產生約 100 萬個小檔案，
請先確認磁碟空間與 inode 數量足夠。

VRAM 不足時，調小推論那節的 `GPU_BATCH`（設 1024 約需 1 GB）與訓練的
`BATCH_SIZE` 即可，兩者都不影響結果，只影響速度。

本 notebook 依賴 Linux 的 `fork` 啟動方式（多進程的 worker 函式定義在 notebook 內）。
在 macOS 或 Windows 上執行時，請把 `NPROC` 設為 1，或把相關函式移到獨立的 `.py`。

執行前請把下一格的路徑改成你自己的資料集位置。


In [ ]:
import glob
import gzip
import json
import os
import time
from multiprocessing import Pool, Process

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset

from utils import SGFParsePlayerIdentification
from network import GoPlayerResNet

print('torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
# 官方資料放置位置：training/ 與 tests/ 依官方發佈的結構擺放
TRAIN_DIR = './training'
TEST_DIR = './tests'
FEAT_DIR = './player-features'
INDEX_PATH = './player_index.json'
CKPT_DIR = './player-trained-models'

NPROC = 8              # 特徵抽取與 SGF 解析的進程數，設成 CPU 核數
MIN_GAMES = 2          # triplet 需要同一位玩家至少兩局（anchor 與 positive 不同局）
SEED = 42

BATCH_SIZE = 100
LEARNING_RATE = 1e-4
MARGIN = 0.5
NUM_EPOCHS = 5
STEPS_PER_EPOCH = 15_000
LOG_INTERVAL = 5_000
TOP_N = 5              # 提交要求 Top-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.makedirs(CKPT_DIR, exist_ok=True)
print('device', DEVICE)
print('總訓練量 %s 步 = %s 個 triplet'
      % (format(NUM_EPOCHS * STEPS_PER_EPOCH, ','),
         format(NUM_EPOCHS * STEPS_PER_EPOCH * BATCH_SIZE, ',')))

# 0. Quick Look to the Dataset

訓練資料與棋力預測子題共用：每個等級一個 CSV，共 10 檔、各 10 萬局，總計 100 萬局。
欄位為 `player_id,game_id,rank,color,sgf_content`，本任務用得到的是 `player_id`
（誰下的）、`color`（執色）與 `sgf_content`（棋譜）。

`player_id` 是本任務的標註來源——它告訴我們哪些棋局出自同一人，訓練時才配得出
「同一人的兩局」與「另一人的一局」。

**過濾條件**：訓練需要同一位玩家至少兩局（一局當 anchor、另一局當 positive），
只有一局的玩家無法組成 triplet，先濾掉。

In [ ]:
df = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{TRAIN_DIR}/train_*.csv'))],
               ignore_index=True)

counts = df['player_id'].value_counts()
keep = set(counts[counts >= MIN_GAMES].index)
dropped = len(counts) - len(keep)
df = df[df['player_id'].isin(keep)].reset_index(drop=True)

print('可用 %s 局 / %s 位玩家（濾掉局數 < %d 的 %d 位）'
      % (format(len(df), ','), format(len(keep), ','), MIN_GAMES, dropped))
print('每位玩家局數：中位數 %d，最多 %d' % (counts[list(keep)].median(),
                                            counts[list(keep)].max()))
df.head(3)

# 1. Feature Extraction

本節是**一次性作業**。特徵產出後重跑訓練可以直接跳到第 2 節。

`SGFParsePlayerIdentification` 重播一局棋，**每下一手就記錄一個窗口**：

```
17 層 = 8 層（自己最近 8 手的盤面）
      + 8 層（對手最近 8 手的盤面）
      + 1 層（執色）
```

所以一局約 100 手就產生約 100 個 `(17, 19, 19)` 的窗口。每個窗口都是一次「簽名」——
模型要學的正是「這一手的下法像不像某個人」。

In [ ]:
_parser = SGFParsePlayerIdentification()
_row = df.iloc[0]
_windows = _parser.extract_features(_row['sgf_content'],
                                    str(_row['color']).lower(), mode='inference')
print('這局產生 %d 個窗口，每個形狀 %s'
      % (len(_windows), np.asarray(_windows[0]).shape))

### 存成一局一個檔

100 萬局的特徵不可能放進記憶體，必須落地。本 notebook 一局存成一個 gzip 壓縮的檔案。

特徵是二元遮罩（每格非 0 即 1），壓縮率約 42 倍——單局解壓後 863 KB，存到磁碟
只有 20 KB。代價是訓練取樣時要多一步：想要某一手，得先解壓整局再取出那一手。

另外產出 `player_index.json` 記錄哪些局屬於哪位玩家——訓練時要先挑玩家才挑局，
沒有這份索引就配不出 triplet。

檔名帶等級前綴（`{rank}_{game_id}`）：`game_id` 只在單一等級的 CSV 內唯一，
10 個檔案都各自從 `g_0000001` 起編，不加前綴會互相覆蓋。


In [ ]:
def extract_worker(df_part, wid, shard_path):
    """抽一份切片的特徵。每個進程獨立持有 parser，並寫自己的索引分片。"""
    parser = SGFParsePlayerIdentification()
    written = bad = 0
    index = {}
    t0 = time.time()

    for _, row in df_part.iterrows():
        key = '%s_%s' % (row['rank'], row['game_id'])
        try:
            windows = parser.extract_features(
                row['sgf_content'], str(row['color']).lower(), mode='inference')
        except Exception:
            bad += 1
            continue
        if not len(windows):
            bad += 1
            continue

        arr = np.asarray(windows, dtype=np.uint8)
        with gzip.open('%s/%s.npy.gz' % (FEAT_DIR, key), 'wb', compresslevel=6) as f:
            np.save(f, arr)
        index.setdefault(row['player_id'], []).append(key)
        written += 1
        if written % 20000 == 0:
            print('  [w%d] %d/%d  %.0f 分'
                  % (wid, written, len(df_part), (time.time() - t0) / 60), flush=True)

    with open(shard_path, 'w') as f:
        json.dump(index, f)
    print('  [w%d] 寫入 %d 局（解析失敗 %d）%.1f 分'
          % (wid, written, bad, (time.time() - t0) / 60), flush=True)


def extract_all(df):
    """多進程抽取，合併索引，並確認產出真的有內容。"""
    os.makedirs(FEAT_DIR, exist_ok=True)

    procs, shards = [], []
    for i in range(NPROC):
        shard = '%s/.index_w%d.json' % (FEAT_DIR, i)
        shards.append(shard)
        p = Process(target=extract_worker, args=(df.iloc[i::NPROC], i, shard))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
        if p.exitcode != 0:
            raise SystemExit('worker 非正常結束（exitcode %s）' % p.exitcode)

    merged = {}
    for shard in shards:
        with open(shard) as f:
            for pid, keys in json.load(f).items():
                merged.setdefault(pid, []).extend(keys)
        os.remove(shard)

    usable = {pid: ks for pid, ks in merged.items() if len(ks) >= MIN_GAMES}
    with open(INDEX_PATH, 'w') as f:
        json.dump(usable, f)

    files = glob.glob(FEAT_DIR + '/*.npy.gz')
    total = sum(os.path.getsize(x) for x in files)
    avg = total / len(files) if files else 0
    print('產出 %s 個檔、%.2f GB、平均 %.1f KB/檔'
          % (format(len(files), ','), total / 1e9, avg / 1024))
    print('索引：%s 位可用玩家' % format(len(usable), ','))
    # 平均檔案大小是最簡單的健全性檢查：寫入迴圈若出錯，計數器仍會累加、log 看起來
    # 正常，但檔案是空的（gzip 後僅 46 bytes）。只數檔案數量抓不到這種情況。
    if avg < 2 * 1024:
        raise SystemExit('平均僅 %.0f bytes，判定為空檔' % avg)

In [ ]:
# 一次性作業。特徵產出後重跑訓練可以直接從下一節開始。
t0 = time.time()
extract_all(df)
print('完成，總耗時 %.1f 分' % ((time.time() - t0) / 60))

# 2. Data Loader

**若已完成第 1 節的特徵抽取，可以直接從這裡開始。**

訓練資料不是「一局棋」，而是**三局棋的組合**：

```
anchor    某位玩家的一手棋      基準
positive  同一位玩家的另一手    要靠近基準
negative  另一位玩家的一手      要遠離基準
```

模型把三者各壓成一個 256 維向量，損失函數要求：

```
loss = max(0,  d(anchor, positive) − d(anchor, negative) + margin)
```

同人的距離要比異人的距離**至少小一個 margin**（0.5）。已經分得夠開的組合 loss 為 0，
不產生梯度，訓練力氣自動集中在還沒分開的組合上。

取樣是三層均勻抽樣：**挑兩位玩家 → 正例挑兩局 → 各取一手**。

### 兩個容易踩到的取樣問題

`IterableDataset` 搭配多個 DataLoader worker 時，每個 worker 會各自執行一次
`__iter__`。如果亂數種子相同，**所有 worker 會抽出完全一樣的 triplet**，等於白費
其他 worker 的算力。

同樣地，每個 epoch 都會重新呼叫 `__iter__`。種子若不隨 epoch 改變，**每個 epoch
會重複抽到同一批題目**——跑 5 個 epoch 只等於把同一批資料看 5 遍，有效訓練量少了
五分之四。

下面的寫法把種子同時綁定 worker 編號與 epoch 計數，避開這兩種重複。

In [ ]:
def load_window(key, rng):
    """載入一局的全部窗口，隨機回傳其中一手。"""
    with gzip.open('%s/%s.npy.gz' % (FEAT_DIR, key), 'rb') as f:
        arr = np.load(f)
    return arr[rng.integers(len(arr))]


class TripletDataset(IterableDataset):
    def __init__(self, index, steps_per_epoch, batch_size):
        self.players = list(index.keys())
        self.index = index
        self.samples = steps_per_epoch * batch_size
        self.epoch = 0

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        wid = info.id if info else 0
        nworkers = info.num_workers if info else 1

        # 種子同時綁 worker 與 epoch，兩個維度都要分開
        rng = np.random.default_rng(SEED + 1000 * self.epoch + wid)
        self.epoch += 1

        for _ in range(self.samples // nworkers):
            pos_player, neg_player = rng.choice(self.players, 2, replace=False)
            pos_games = self.index[pos_player]
            neg_games = self.index[neg_player]
            a_key, p_key = rng.choice(pos_games, 2, replace=False)
            n_key = neg_games[rng.integers(len(neg_games))]
            try:
                yield (load_window(a_key, rng),
                       load_window(p_key, rng),
                       load_window(n_key, rng))
            except (OSError, EOFError, ValueError):
                continue          # 容忍缺檔，跳過該筆

In [ ]:
with open(INDEX_PATH) as f:
    index = json.load(f)
print('索引：%s 位玩家 / %s 局'
      % (format(len(index), ','), format(sum(len(v) for v in index.values()), ',')))

train_dataset = TripletDataset(index, STEPS_PER_EPOCH, BATCH_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NPROC,
                          pin_memory=True, persistent_workers=True, drop_last=True)

_a, _p, _n = next(iter(train_loader))
print('一個 batch:', _a.shape, _p.shape, _n.shape)

# 3. Model Training

`GoPlayerResNet` 的輸出是 256 維向量，並經過 **L2 正規化**（長度固定為 1）。

正規化這步很關鍵：沒有它，模型可以靠「把所有向量放大」讓所有距離都超過 margin，
loss 降到 0 卻什麼都沒學到。固定長度之後，距離只反映方向差異，模型只能真的去學
「什麼特徵能區分兩個人」。

一個實作細節：anchor / positive / negative 可以串成一個 batch 過一次模型，
再切回三份算 loss。搭配 bf16 混合精度後，實測比三次獨立 forward 快約一倍。

In [ ]:
model = GoPlayerResNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.TripletMarginLoss(margin=MARGIN, p=2, reduction='mean')

print('參數量 {:,}'.format(sum(p.numel() for p in model.parameters())))
print('batch %d | lr %s | margin %s | %d epochs × %s 步'
      % (BATCH_SIZE, LEARNING_RATE, MARGIN, NUM_EPOCHS,
         format(STEPS_PER_EPOCH, ',')))

In [ ]:
def train_step(model, optimizer, criterion, batch):
    """一個 triplet step。三路合併成單次 forward，以 bf16 執行。"""
    anchor, positive, negative = batch
    n = anchor.size(0)
    # uint8 送 GPU 再轉 float：傳輸量是先轉 float32 的四分之一
    x = torch.cat([anchor, positive, negative], 0).to(DEVICE, non_blocking=True)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=DEVICE.type == 'cuda'):
        a, p, neg = model(x.float()).split(n)
        loss = criterion(a, p, neg)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
# 訓練期間不能中斷 kernel。
history = []
model.train()
best = float('inf')

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    run_loss, steps = 0.0, 0

    for batch in train_loader:
        run_loss += train_step(model, optimizer, criterion, batch)
        steps += 1
        if steps % LOG_INTERVAL == 0:
            print('  ep%d step %s  loss %.4f  %.1f 分'
                  % (epoch + 1, format(steps, ','), run_loss / steps,
                     (time.time() - t0) / 60), flush=True)
        if steps >= STEPS_PER_EPOCH:
            break

    avg = run_loss / max(steps, 1)
    history.append({'epoch': epoch + 1, 'loss': avg,
                    'minutes': (time.time() - t0) / 60})
    print('Epoch %d/%d  loss %.4f  %.1f 分'
          % (epoch + 1, NUM_EPOCHS, avg, (time.time() - t0) / 60), flush=True)

    if avg < best:
        best = avg
        torch.save(model.state_dict(), f'{CKPT_DIR}/best_model.pth')
        print('  新的最低 loss %.4f —— 已存 best_model.pth' % best)
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_%d.pth' % (epoch + 1))

pd.DataFrame(history).to_csv(f'{CKPT_DIR}/history.csv', index=False)
print('訓練完成，最低 loss %.4f' % best)

### 怎麼判斷訓練有沒有在進行

初始 loss 應該在 **margin（0.5）附近**——未訓練的模型輸出接近隨機投影，
`d(anchor, positive)` 與 `d(anchor, negative)` 差不多大，兩者相減約為 0，
loss 就等於 margin。

訓練有效的話 loss 會穩定往下。若第一個 epoch 結束仍貼在 0.5 不動，代表模型完全沒學
到東西，繼續跑也不會好轉——這種情況換一個 seed 或降低學習率再試。

本 notebook 實際跑出來的曲線：

In [ ]:
import matplotlib.pyplot as plt

hist = pd.read_csv(f'{CKPT_DIR}/history.csv')

plt.figure(figsize=(8, 4))
plt.plot(hist['epoch'], hist['loss'], marker='o')
plt.axhline(MARGIN, ls='--', c='gray', lw=1)
plt.annotate('margin = %.1f (untrained level)' % MARGIN,
             xy=(hist['epoch'].iloc[0], MARGIN), xytext=(1.2, MARGIN - 0.02),
             color='gray')
plt.xlabel('epoch'); plt.ylabel('triplet loss'); plt.xticks(list(hist['epoch']))
plt.ylim(0, 0.55); plt.grid(alpha=.3); plt.tight_layout(); plt.show()


# 4. Inference with Testing Data On-the-Fly

考試的流程分兩步：

**建立候選庫的指紋。** 400 位候選玩家，每人約 100 局。把這些棋全部過一次模型，
每局的每個窗口都得到一個 256 維向量，**全部取平均**，得到這位玩家的代表向量
（重心）。

**比對查詢。** 每道題的棋譜同樣算出平均向量，跟 400 個重心比 L2 距離，交出最近的
5 個。

取平均這步是分數的主要來源。單一手棋能承載的個人資訊很少，但一百局平均下來，
隨機性被抵消，剩下的就是這個人穩定的傾向。

**Public 與 Private 兩份題目都要預測，串接成一份 800 列的 submission 上傳。**
兩池的候選庫是分開的——`pub_` 開頭的題目只能填 `pub_p_` 開頭的玩家，填錯池會被判
無效提交。

In [ ]:
GPU_BATCH = 4096
CHUNK = 256
_infer_parser = SGFParsePlayerIdentification()


def sgf_windows(args):
    """給 multiprocessing.Pool 用的頂層函式。(sgf, color) → (N, 17, 19, 19)"""
    sgf, color = args
    try:
        feats = _infer_parser.extract_features(sgf, str(color).lower(),
                                               mode='inference')
        return np.asarray(feats, dtype=np.uint8) if len(feats) else None
    except Exception:
        return None


def embed(model, arr):
    outs = []
    with torch.no_grad():
        for i in range(0, len(arr), GPU_BATCH):
            x = torch.from_numpy(arr[i:i + GPU_BATCH]).float().to(DEVICE)
            outs.append(model(x).cpu().numpy())
    return np.concatenate(outs) if outs else np.zeros((0, 256), np.float32)


def mean_embeddings(model, pool, jobs, keys, label):
    """jobs 與 keys 等長；把同一個 key 的所有窗口平均成一個向量。"""
    sums, counts = {}, {}
    t0 = time.time()
    for start in range(0, len(jobs), CHUNK):
        for key, arr in zip(keys[start:start + CHUNK],
                            pool.map(sgf_windows, jobs[start:start + CHUNK])):
            if arr is None:
                continue
            e = embed(model, arr)
            sums[key] = sums.get(key, 0) + e.sum(0)
            counts[key] = counts.get(key, 0) + len(e)
        done = min(start + CHUNK, len(jobs))
        if done % (CHUNK * 20) == 0 or done == len(jobs):
            el = time.time() - t0
            print('  %s %d/%d 局  %.0f 秒' % (label, done, len(jobs), el), flush=True)
    return {k: sums[k] / counts[k] for k in sums}


def predict(weights_path, out_path):
    net = GoPlayerResNet()
    net.load_state_dict(torch.load(weights_path, map_location='cpu'))
    net.eval().to(DEVICE)

    rows = []
    with Pool(NPROC) as pool:
        # 兩池各自獨立：候選庫不同，不能混用
        for split in ('public', 'private'):
            print('=== %s ===' % split, flush=True)
            cand = pd.read_csv(
                f'{TEST_DIR}/player_identification_candidates_{split}.csv')
            centroids = mean_embeddings(
                net, pool, list(zip(cand['sgf_content'], cand['color'])),
                list(cand['player_id']), '候選')
            names = sorted(centroids)
            C = np.stack([centroids[n] for n in names])
            print('  重心 %d 個' % len(names), flush=True)

            test = pd.read_csv(f'{TEST_DIR}/player_identification_test_{split}.csv')
            sgf_cols = [c for c in test.columns if c.startswith('sgf_')]
            col_cols = [c for c in test.columns if c.startswith('color_')]

            jobs, keys = [], []
            for _, r in test.iterrows():
                # num_games 之後的 sgf_ 欄位是空字串，要濾掉
                for sc, cc in zip(sgf_cols, col_cols):
                    if pd.notna(r[sc]) and str(r[sc]).strip():
                        jobs.append((r[sc], r[cc]))
                        keys.append(r['question_id'])
            queries = mean_embeddings(net, pool, jobs, keys, '查詢')

            for qid in test['question_id']:
                q = queries.get(qid)
                if q is None:
                    rows.append([qid] + names[:TOP_N])
                    continue
                d = np.linalg.norm(C - q, axis=1)
                rows.append([qid] + [names[i] for i in np.argsort(d)[:TOP_N]])
            print('  %s 完成 %d 題' % (split, len(test)), flush=True)

    sub = pd.DataFrame(rows,
                       columns=['question_id'] + ['top_%d' % i
                                                  for i in range(1, TOP_N + 1)])
    sub.to_csv(out_path, index=False)
    print('寫出 %s（%d 列）' % (out_path, len(sub)))
    return sub

In [ ]:
submission = predict(f'{CKPT_DIR}/best_model.pth', './submission_player.csv')
submission.head()

# End of the tutorial

玩家辨識 tutorial 到此結束。照本 notebook 從頭跑完一次，在 Public 測試集上可以得到
**0.2845** 的分數。

Private 測試集的分數於競賽結束後才公布，所以這裡不列——排行榜上看得到的、
能拿來比較自己改動的，只有 Public 分數。

這個分數是在下列環境跑出來的：

| 項目 | 規格 |
|---|---|
| GPU | NVIDIA GeForce RTX 4090（24 GB，driver 535.288.01） |
| CPU | AMD EPYC-Genoa，8 核 |
| 記憶體 | 31 GB |
| OS | Ubuntu 22.04.5 LTS |
| 環境 | Python 3.10.12、PyTorch 2.6.0+cu124、CUDA 12.4 |

本 tutorial 僅供參考，本競賽對模型架構、特徵設計與訓練方式都沒有限制，
歡迎依需求修改與改進。祝你在玩家辨識模型上一切順利！
